# From Raw Data to Corpus, Retrieval Artifacts, and Metrics

This notebook is a learning guide for the current PubMedQA/SciFact retrieval pipeline.

The goal is to make the moving pieces explicit:

```text
raw dataset rows
-> query rows
-> pooled corpus passages
-> retriever top-k results
-> normalized v0.1 artifact rows
-> metric summary
```

Important: answer generation is still not the focus here. The retrieval artifacts use gold-label demo predictions so we can isolate retrieval and metric mechanics first.

## 1. Setup

Run this notebook from the `rag_project/` repo root. The code uses the local project package and writes temporary artifacts under ignored `outputs/`.

In [11]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

from rag_experiment.corpus.pubmedqa import (
    build_pubmedqa_passages,
    load_pubmedqa_rows,
    pubmedqa_gold_evidence,
    select_pubmedqa_corpus_rows,
    select_pubmedqa_queries,
)
from rag_experiment.corpus.scifact import (
    build_scifact_passages,
    load_scifact_data,
    scifact_gold_evidence,
    scifact_label,
    select_labeled_claims,
    select_scifact_corpus_doc_ids,
)
from rag_experiment.data.inspect_datasets import DEFAULT_HF_CACHE, DEFAULT_SCIFACT_DIR
from rag_experiment.evaluation.evaluate_artifact import evaluate_file
from rag_experiment.runners.run_pubmedqa_retrieval import run_pubmedqa_retrieval
from rag_experiment.runners.run_scifact_retrieval import run_scifact_retrieval
from rag_experiment.runners.pooled_retrieval import pooled_output_path

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'retrieval'

PROJECT_ROOT

PosixPath('/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/notebooks')

## 2. The Mental Model

The project has a few names that sound similar but mean different things.

| Name | Meaning | Example |
|---|---|---|
| raw row | One original dataset item | PubMedQA row or SciFact claim/corpus row |
| query row | The item we evaluate | A PubMedQA question or SciFact claim |
| corpus passage | Text the retriever is allowed to search | PubMedQA context section or SciFact abstract sentence |
| gold evidence | Key(s) that count as correct retrieval | `(pubid, context_idx)` or `(doc_id, sentence_index)` |
| retrieved passage | One top-k result returned by BM25/dense/hybrid | Saved under `retrieved_passages` |
| prediction | Answer field for answer metrics | Gold-label demo for now, real model later |

The important separation is:

```text
query limit = how many examples we evaluate
corpus limit = how many source docs/rows the retriever searches over
```

If the corpus is too small to contain the evaluated query's gold context, retrieval metrics become unfair. That is why PubMedQA rejects nonzero `corpus_limit < limit`.

## 3. PubMedQA: Raw Rows to Corpus Passages

PubMedQA gives one row per biomedical yes/no/maybe question. The retrievable text should come from `context.contexts` only.

Do **not** put these fields into corpus passage text:

- `question`: makes retrieval too easy by matching the query to itself
- `long_answer`: often acts like the explanation/conclusion
- `final_decision`: directly leaks yes/no/maybe
- any model prediction field

Those fields may appear in example metadata for analysis, but they should not be searchable passage text.

In [26]:
pub_rows = load_pubmedqa_rows(cache_dir=DEFAULT_HF_CACHE)
print('usable PubMedQA rows:', len(pub_rows))

first_pub = pub_rows[0]
print('raw keys:', sorted(first_pub.keys()))
print('pubid:', first_pub['pubid'])
print('question:', first_pub['question'])
print('final_decision:', first_pub['final_decision'])
print('context labels:', first_pub['context'].get('labels'))
print('context count:', len(first_pub['context'].get('contexts') or []))

usable PubMedQA rows: 1000
raw keys: ['context', 'final_decision', 'long_answer', 'pubid', 'question']
pubid: 21645374
question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
final_decision: yes
context labels: ['BACKGROUND', 'RESULTS']
context count: 2


In [27]:
sec_pub = pub_rows[1]
print('raw keys:', sorted(sec_pub.keys()))
print('pubid:', sec_pub['pubid'])
print('question:', sec_pub['question'])
print('final_decision:', sec_pub['final_decision'])
print('context labels:', sec_pub['context'].get('labels'))
print('context count:', len(sec_pub['context'].get('contexts') or []))

raw keys: ['context', 'final_decision', 'long_answer', 'pubid', 'question']
pubid: 16418930
question: Landolt C and snellen e acuity: differences in strabismus amblyopia?
final_decision: no
context labels: ['BACKGROUND', 'PATIENTS AND METHODS', 'RESULTS']
context count: 3


In [13]:
PUB_QUERY_LIMIT = 5
PUB_CORPUS_LIMIT = 20

pub_query_rows = select_pubmedqa_queries(pub_rows, limit=PUB_QUERY_LIMIT)
pub_corpus_rows = select_pubmedqa_corpus_rows(
    pub_rows,
    corpus_limit=PUB_CORPUS_LIMIT,
    query_limit=PUB_QUERY_LIMIT,
)
pub_passages = build_pubmedqa_passages(pub_corpus_rows)

print('query rows:', len(pub_query_rows))
print('corpus rows:', len(pub_corpus_rows))
print('corpus passages:', len(pub_passages))

display(pd.DataFrame([
    {
        'passage_id': p.id,
        'pubid': p.example_id,
        'context_idx': p.sentence_index,
        'section': p.title,
        'text_preview': ' '.join(p.text.split())[:180],
    }
    for p in pub_passages[:10]
]))

query rows: 5
corpus rows: 20
corpus passages: 64


,passage_id,pubid,context_idx,section,text_preview
0,pubmedqa::21645374::0,21645374,0,BACKGROUND,Programmed cell death (PCD) is the regulated d...
1,pubmedqa::21645374::1,21645374,1,RESULTS,The following paper elucidates the role of mit...
2,pubmedqa::16418930::0,16418930,0,BACKGROUND,Assessment of visual acuity depends on the opt...
3,pubmedqa::16418930::1,16418930,1,PATIENTS AND METHODS,"100 patients (age 8 - 90 years, median 60.5 ye..."
4,pubmedqa::16418930::2,16418930,2,RESULTS,Differences between Landolt C acuity (LR) and ...
5,pubmedqa::9488747::0,9488747,0,BACKGROUND,Apparent life-threatening events in infants ar...
6,pubmedqa::9488747::1,9488747,1,CASE REPORTS,Eight infants aged 2 to 15 months were admitte...
7,pubmedqa::17208539::0,17208539,0,PURPOSE,The transanal endorectal pull-through (TERPT) ...
8,pubmedqa::17208539::1,17208539,1,METHODS,Records of 41 patients more than 3 years old w...
9,pubmedqa::17208539::2,17208539,2,RESULTS,"Overall scores were similar. However, continen..."


### Why PubMedQA Rejects `corpus_limit < limit`

The first `limit` rows are evaluated as queries. If `corpus_limit` is smaller than that, some evaluated questions have gold contexts that are not in the retrieval corpus. The retriever would be punished for not returning passages it was never allowed to see.

In [14]:
try:
    select_pubmedqa_corpus_rows(pub_rows, corpus_limit=3, query_limit=5)
except ValueError as exc:
    print(type(exc).__name__)
    print(exc)

ValueError
PubMedQA corpus_limit must be 0 or >= limit so every evaluated query has its gold context in the retrieval corpus


## 4. SciFact: Claims + Separate Corpus

SciFact is structurally cleaner for evidence retrieval because claims and corpus documents are separate files.

- Claims provide the query text, label, and gold rationale sentence indices.
- Corpus rows provide document titles and abstract sentences.
- Retrieval passages are sentence-level: one abstract sentence becomes one passage.

This is why SciFact metrics can check exact `(doc_id, sentence_index)` evidence, while PubMedQA can only do coarser context checks.

In [15]:
scifact_corpus, scifact_claims = load_scifact_data(data_dir=DEFAULT_SCIFACT_DIR)
print('SciFact corpus docs:', len(scifact_corpus))
print('SciFact train claims:', len(scifact_claims))

first_claim = select_labeled_claims(scifact_claims, corpus=scifact_corpus, limit=1)[0]
print('claim id:', first_claim['id'])
print('claim:', first_claim['claim'])
print('label:', scifact_label(first_claim))
print('gold evidence:', scifact_gold_evidence(first_claim, scifact_corpus))

SciFact corpus docs: 5183
SciFact train claims: 809
claim id: 2
claim: 1 in 5 million in UK have abnormal PrP positivity.
label: CONTRADICT
gold evidence: [{'doc_id': '13734012', 'sentence_index': 4}]


In [16]:
SCI_QUERY_LIMIT = 5
SCI_CORPUS_DOC_LIMIT = 50

sci_query_rows = select_labeled_claims(
    scifact_claims,
    corpus=scifact_corpus,
    limit=SCI_QUERY_LIMIT,
)
sci_doc_ids = select_scifact_corpus_doc_ids(
    corpus=scifact_corpus,
    claims=sci_query_rows,
    corpus_doc_limit=SCI_CORPUS_DOC_LIMIT,
)
sci_passages = build_scifact_passages(corpus=scifact_corpus, doc_ids=sci_doc_ids)

print('query claims:', len(sci_query_rows))
print('corpus docs:', len(sci_doc_ids))
print('sentence passages:', len(sci_passages))

display(pd.DataFrame([
    {
        'passage_id': p.id,
        'doc_id': p.example_id,
        'sentence_index': p.sentence_index,
        'title': p.title,
        'text_preview': ' '.join(p.text.split())[:180],
    }
    for p in sci_passages[:5]
]))

query claims: 5
corpus docs: 50
sentence passages: 456


,passage_id,doc_id,sentence_index,title,text_preview
0,scifact::6490571::0,6490571,0,"Prevalence, severity, and unmet need for treat...",CONTEXT Little is known about the extent or se...
1,scifact::6490571::1,6490571,1,"Prevalence, severity, and unmet need for treat...","OBJECTIVE To estimate prevalence, severity, an..."
2,scifact::6490571::2,6490571,2,"Prevalence, severity, and unmet need for treat...","DESIGN, SETTING, AND PARTICIPANTS Face-to-face..."
3,scifact::6490571::3,6490571,3,"Prevalence, severity, and unmet need for treat...","MAIN OUTCOME MEASURES The DSM-IV disorders, se..."
4,scifact::6490571::4,6490571,4,"Prevalence, severity, and unmet need for treat...",RESULTS The prevalence of having any WMH-CIDI/...


## 5. Raw Fields to Normalized Artifact Fields

The normalized artifact is the common shape the evaluator understands. PubMedQA and SciFact have different raw schemas, but both are converted into this shape.

| Artifact field | PubMedQA source | SciFact source |
|---|---|---|
| `dataset` | literal `pubmedqa` | literal `scifact` |
| `example.id` | `pubid` | claim `id` |
| `example.query` | `question` | `claim` |
| `example.gold_answer` | `final_decision` | evidence label |
| `example.gold_evidence` | all `(pubid, context_idx)` for this row | rationale `(doc_id, sentence_index)` |
| `retrieved_passages[].text` | retrieved context section | retrieved abstract sentence |
| `retrieved_passages[].metadata` | `pubid`, `context_idx`, label | `doc_id`, `sentence_index`, title |

For retrieval-only artifacts, `prediction.answer` is still the gold label demo. That makes answer accuracy uninteresting for now, but lets retrieval metrics run.

## 6. Run Pooled BM25 Retrieval

BM25 is the easiest retrieval method to learn from because it does not call an API.

These calls write normalized artifacts under `outputs/retrieval/`. They are small enough to inspect manually.

In [17]:
PUB_BM25_PATH = pooled_output_path('pubmedqa', 'bm25')
SCI_BM25_PATH = pooled_output_path('scifact', 'bm25')

run_pubmedqa_retrieval(
    retriever_name='bm25',
    limit=20,
    corpus_limit=20,
    top_k=5,
    output_path=PUB_BM25_PATH,
    cache_dir=DEFAULT_HF_CACHE,
)

run_scifact_retrieval(
    retriever_name='bm25',
    limit=20,
    corpus_doc_limit=300,
    top_k=5,
    output_path=SCI_BM25_PATH,
    data_dir=DEFAULT_SCIFACT_DIR,
)

print(PUB_BM25_PATH)
print(SCI_BM25_PATH)

/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/retrieval/pubmedqa_bm25_pooled_v01.jsonl
/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/retrieval/scifact_bm25_pooled_v01.jsonl


## 7. Load and Inspect One Artifact Row

A retrieval artifact is JSONL: one JSON object per evaluated query. Start by reading one row and identifying the major sections.

In [18]:
def load_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

pub_artifact = load_jsonl(PUB_BM25_PATH)
sci_artifact = load_jsonl(SCI_BM25_PATH)

row = pub_artifact[0]
print('top-level keys:', row.keys())
print('run:', row['run'])
print('example keys:', row['example'].keys())
print('retrieved passage count:', len(row['retrieved_passages']))
print('prediction:', row['prediction'])

top-level keys: dict_keys(['schema_version', 'dataset', 'run', 'example', 'retrieved_passages', 'prediction', 'error'])
run: {'name': 'pubmedqa_bm25_pooled', 'retriever': 'bm25', 'top_k': 5, 'model_profile': 'gold_label_demo', 'sample_limit': 20, 'corpus_limit': 20, 'corpus_row_count': 20, 'passage_count': 64}
example keys: dict_keys(['id', 'query', 'gold_answer', 'gold_evidence', 'metadata'])
retrieved passage count: 5
prediction: {'answer': 'yes', 'cited_passage_ids': [], 'metadata': {'source': 'gold_label_demo'}}


In [19]:
def preview_text(text, length=160):
    text = ' '.join(str(text).split())
    return text if len(text) <= length else text[:length] + '...'

pub_row = pub_artifact[0]
print('PubMedQA query:', pub_row['example']['query'])
print('gold evidence:', pub_row['example']['gold_evidence'])

display(pd.DataFrame([
    {
        'rank': p['rank'],
        'pubid': p['metadata']['pubid'],
        'context_idx': p['metadata']['context_idx'],
        'section': p['metadata'].get('context_label'),
        'is_gold_pubid': p['metadata']['pubid'] == pub_row['example']['id'],
        'text_preview': preview_text(p['text']),
    }
    for p in pub_row['retrieved_passages']
]))

PubMedQA query: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
gold evidence: [{'pubid': '21645374', 'context_idx': 0}, {'pubid': '21645374', 'context_idx': 1}]


,rank,pubid,context_idx,section,is_gold_pubid,text_preview
0,1,21645374,0,BACKGROUND,True,Programmed cell death (PCD) is the regulated d...
1,2,21645374,1,RESULTS,True,The following paper elucidates the role of mit...
2,3,18239988,0,AIMS,False,Specific markers for differentiation of nonalc...
3,4,10966337,1,METHODS,False,This is a descriptive comparison of prospectiv...
4,5,9488747,1,CASE REPORTS,False,Eight infants aged 2 to 15 months were admitte...


In [20]:
sci_row = sci_artifact[0]
print('SciFact claim:', sci_row['example']['query'])
print('gold evidence:', sci_row['example']['gold_evidence'])

display(pd.DataFrame([
    {
        'rank': p['rank'],
        'doc_id': p['metadata']['doc_id'],
        'sentence_index': p['metadata']['sentence_index'],
        'is_gold_sentence': {
            'doc_id': p['metadata']['doc_id'],
            'sentence_index': p['metadata']['sentence_index'],
        } in sci_row['example']['gold_evidence'],
        'text_preview': preview_text(p['text']),
    }
    for p in sci_row['retrieved_passages']
]))

SciFact claim: 1 in 5 million in UK have abnormal PrP positivity.
gold evidence: [{'doc_id': '13734012', 'sentence_index': 4}]


,rank,doc_id,sentence_index,is_gold_sentence,text_preview
0,1,409280,7,False,Fewer than 1 in 5 physicians knew that more wo...
1,2,463309,3,False,"The optimum incubation time was 1 h, and the o..."
2,3,13734012,4,True,"RESULTS Of the 32,441 appendix samples 16 were..."
3,4,11705328,2,False,METHODS We randomized 151 patients with ischem...
4,5,374902,0,False,BACKGROUND Roughly 3 million people worldwide ...


## 8. Compute the Core Retrieval Checks by Hand

The evaluator mostly compares small key tuples.

For PubMedQA:

```text
gold key = (pubid, context_idx)
retrieved key = (metadata.pubid, metadata.context_idx)
```

For SciFact:

```text
gold key = (doc_id, sentence_index)
retrieved key = (metadata.doc_id, metadata.sentence_index)
```

This is the main thing to understand before worrying about metric names.

In [21]:
def pubmedqa_keys_from_gold(items):
    return {(str(item['pubid']), int(item['context_idx'])) for item in items}

def pubmedqa_keys_from_retrieved(passages):
    return {
        (str(p['metadata']['pubid']), int(p['metadata']['context_idx']))
        for p in passages
    }

pub_gold = pubmedqa_keys_from_gold(pub_row['example']['gold_evidence'])
pub_retrieved = pubmedqa_keys_from_retrieved(pub_row['retrieved_passages'])
pub_matched = pub_gold & pub_retrieved

print('gold:', pub_gold)
print('retrieved:', pub_retrieved)
print('matched:', pub_matched)
print('hit for this row:', bool(pub_matched))
print('recall for this row:', len(pub_matched) / len(pub_gold))

gold: {('21645374', 0), ('21645374', 1)}
retrieved: {('10966337', 1), ('21645374', 1), ('18239988', 0), ('21645374', 0), ('9488747', 1)}
matched: {('21645374', 0), ('21645374', 1)}
hit for this row: True
recall for this row: 1.0


In [22]:
def scifact_keys_from_gold(items):
    return {(str(item['doc_id']), int(item['sentence_index'])) for item in items}

def scifact_keys_from_retrieved(passages):
    return {
        (str(p['metadata']['doc_id']), int(p['metadata']['sentence_index']))
        for p in passages
    }

sci_gold = scifact_keys_from_gold(sci_row['example']['gold_evidence'])
sci_retrieved = scifact_keys_from_retrieved(sci_row['retrieved_passages'])
sci_matched = sci_gold & sci_retrieved

print('gold:', sci_gold)
print('retrieved:', sci_retrieved)
print('matched:', sci_matched)
print('sentence hit for this row:', bool(sci_matched))
print('sentence recall for this row:', len(sci_matched) / len(sci_gold))
print('all evidence hit for this row:', len(sci_matched) == len(sci_gold))

gold: {('13734012', 4)}
retrieved: {('11705328', 2), ('374902', 0), ('13734012', 4), ('409280', 7), ('463309', 3)}
matched: {('13734012', 4)}
sentence hit for this row: True
sentence recall for this row: 1.0
all evidence hit for this row: True


## 9. Run the Evaluator

Now the aggregate metrics should be less mysterious: they are the same key comparisons repeated across rows.

Remember: `answer_accuracy` / `label_accuracy` is perfect in these retrieval-only artifacts because `prediction.answer` is still copied from the gold label. The meaningful numbers right now are the retrieval metrics.

In [23]:
pub_summary = evaluate_file(PUB_BM25_PATH, dataset='pubmedqa')
sci_summary = evaluate_file(SCI_BM25_PATH, dataset='scifact')

Normalized Artifact Evaluation
schema_version: v0.1
dataset: pubmedqa
artifact: /Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/retrieval/pubmedqa_bm25_pooled_v01.jsonl
run: {'name': 'pubmedqa_bm25_pooled', 'retriever': 'bm25', 'top_k': 5, 'model_profile': 'gold_label_demo', 'sample_limit': 20, 'corpus_limit': 20, 'corpus_row_count': 20, 'passage_count': 64}
rows: 20  errors: 0
metrics:
  answer_accuracy: 1.0
  answer_correct: 20
  answer_total: 20
  context_hit_at_k: 0.95
  context_hit_count: 19
  context_total: 20
  context_recall_at_k: 0.670833
  context_skipped: 0
Normalized Artifact Evaluation
schema_version: v0.1
dataset: scifact
artifact: /Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project/outputs/retrieval/scifact_bm25_pooled_v01.jsonl
run: {'name': 'scifact_bm25_pooled', 'retriever': 'bm25', 'top_k': 5, 'model_profile': 'gold_label

In [24]:
def metrics_table(summary):
    return pd.DataFrame([
        {'metric': key, 'value': value}
        for key, value in summary['metrics'].items()
    ])

print('PubMedQA metrics')
display(metrics_table(pub_summary))

print('SciFact metrics')
display(metrics_table(sci_summary))

PubMedQA metrics


,metric,value
0,answer_accuracy,1.000000
1,answer_correct,20.000000
2,answer_total,20.000000
3,context_hit_at_k,0.950000
4,context_hit_count,19.000000
5,context_total,20.000000
6,context_recall_at_k,0.670833
7,context_skipped,0.000000


SciFact metrics


,metric,value
0,label_accuracy,1.000
1,label_correct,20.000
2,label_total,20.000
3,evidence_doc_hit_at_k,0.950
4,evidence_doc_hit_count,19.000
5,evidence_sentence_hit_at_k,0.750
6,evidence_sentence_hit_count,15.000
7,evidence_all_hit_at_k,0.450
8,evidence_all_hit_count,9.000
9,evidence_recall_at_k,0.545


## 10. Dense and Hybrid Retrieval

Dense and hybrid use the same artifact and metric shape. The only difference is how `retrieved_passages` are chosen.

- `bm25`: keyword/statistical text matching, no API
- `dense`: embedding similarity, calls DashScope embeddings
- `hybrid`: combines BM25 and dense with ensemble fusion

Keep these runs tiny while learning because dense/hybrid call the embedding API. The project caps DashScope embedding batch size at 25 texts.

In [25]:
RUN_API_BACKED_RETRIEVERS = False

if RUN_API_BACKED_RETRIEVERS:
    pub_dense_path = pooled_output_path('pubmedqa', 'dense')
    pub_hybrid_path = pooled_output_path('pubmedqa', 'hybrid')

    run_pubmedqa_retrieval(
        retriever_name='dense',
        limit=5,
        corpus_limit=20,
        top_k=5,
        output_path=pub_dense_path,
        cache_dir=DEFAULT_HF_CACHE,
    )
    run_pubmedqa_retrieval(
        retriever_name='hybrid',
        limit=5,
        corpus_limit=20,
        top_k=5,
        output_path=pub_hybrid_path,
        cache_dir=DEFAULT_HF_CACHE,
    )

    dense_summary = evaluate_file(pub_dense_path, dataset='pubmedqa')
    hybrid_summary = evaluate_file(pub_hybrid_path, dataset='pubmedqa')
    display(pd.concat([
        metrics_table(dense_summary).assign(retriever='dense'),
        metrics_table(hybrid_summary).assign(retriever='hybrid'),
    ]))
else:
    print('Skipped API-backed dense/hybrid cells. Set RUN_API_BACKED_RETRIEVERS = True to run them.')

Skipped API-backed dense/hybrid cells. Set RUN_API_BACKED_RETRIEVERS = True to run them.


## 11. Noise Reduction Checklist

When you feel lost, classify each field into one of these buckets:

| Bucket | Question to ask |
|---|---|
| raw dataset | Did this come from PubMedQA/SciFact directly? |
| corpus passage | Is this searchable text for the retriever? |
| query/example | Is this the item being evaluated? |
| gold evidence | What key counts as correct retrieval? |
| retrieved passage | What did the retriever return? |
| prediction | What answer did the model/demo provide? |
| metric | Which gold keys are compared to which retrieved keys? |

The current project is mostly about making those buckets explicit and saving them in inspectable JSONL artifacts.

## 12. What This Notebook Proves

This notebook proves that:

- raw PubMedQA/SciFact rows can be turned into pooled retrieval corpora
- query limits and corpus limits are separate concepts
- PubMedQA avoids answer leakage by only indexing `context.contexts`
- normalized artifacts store enough information to inspect retrieval row by row
- metrics are mostly set comparisons between gold evidence keys and retrieved metadata keys
- BM25/dense/hybrid share the same artifact and metric contract

This notebook still does **not** prove answer-generation quality. That comes later, after retrieval behavior is understood.